In [ ]:
!pip install deep-translator

In [ ]:
from pathlib import Path
from deep_translator import GoogleTranslator


def translate_srt_file(
    input_srt: str,
    output_srt: str = None,
    source_lang: str = "fr",
    target_lang: str = "en",
):
    """
    Translate subtitle text in an .srt file while preserving numbering and timestamps.

    Parameters:
        input_srt: path to the original subtitle file
        output_srt: path for the translated subtitle file
        source_lang: source language code, e.g. 'fr'
        target_lang: target language code, e.g. 'en'
    """
    input_path = Path(input_srt)

    if not input_path.exists():
        raise FileNotFoundError(f"Could not find file: {input_srt}")

    if output_srt is None:
        output_srt = str(input_path.with_name(input_path.stem + "_english.srt"))

    translator = GoogleTranslator(source=source_lang, target=target_lang)

    with open(input_path, "r", encoding="utf-8-sig") as f:
        lines = f.readlines()

    translated_lines = []

    for line in lines:
        stripped = line.strip()

        # keep blank lines
        if stripped == "":
            translated_lines.append(line)
            continue

        # keep subtitle numbering
        if stripped.isdigit():
            translated_lines.append(line)
            continue

        # keep timestamp lines
        if "-->" in stripped:
            translated_lines.append(line)
            continue

        # translate subtitle text lines only
        try:
            translated_text = translator.translate(stripped)
            translated_lines.append(translated_text + "\n")
        except Exception as e:
            print(f"Warning: could not translate line: {stripped}")
            print(f"Reason: {e}")
            translated_lines.append(line)

    with open(output_srt, "w", encoding="utf-8") as f:
        f.writelines(translated_lines)

    print(f"Translated subtitles saved to: {output_srt}")
    return output_srt

In [ ]:
output_file = translate_srt_file(
    input_srt="french_subtitles.srt",
    source_lang="fr",
    target_lang="en"
)

output_file